# CatalystIQ Data Exploration - End to End (Free-Tier Friendly)

Uses only currently entitled endpoints from your keys.

## 0) Setup

- Ensure `data-exploration/.env` contains valid keys.
- Run cells top-to-bottom.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import pandas as pd
import plotly.express as px

from src.common import (
    load_config, new_pull_id,
    polygon_grouped_daily, polygon_intraday_bars,
    finnhub_quote, normalize_finnhub_quote,
    quality_summary, save_raw_json, save_parquet
)

# robust local .env parser (avoids BOM/key-name issues)
vals = {}
for line in Path('../.env').read_text(encoding='utf-8').replace('\ufeff','').splitlines():
    if '=' in line:
        k, v = line.split('=', 1)
        vals[k.strip()] = v.strip()

cfg = load_config('../config/exploration.yaml')
poly_key = vals.get('POLYGON_API_KEY', '')
fh_key = vals.get('FINNHUB_API_KEY', '')
poly_base = vals.get('POLYGON_BASE_URL', 'https://api.polygon.io')
fh_base = vals.get('FINNHUB_BASE_URL', 'https://finnhub.io/api/v1')

assert poly_key, 'POLYGON_API_KEY missing'


## 1) Polygon Daily Grouped (free-tier accessible)

Pull last 2 completed trading days and build top movers from day-over-day close change.

In [ ]:
today_et = datetime.now(tz=ZoneInfo('America/New_York')).date()
# simple lookback for 2 completed sessions
d2 = today_et - timedelta(days=1)
d1 = today_et - timedelta(days=2)

pull_daily = new_pull_id('polygon_grouped_daily')

p1 = polygon_grouped_daily(poly_key, poly_base, d1.isoformat())
p2 = polygon_grouped_daily(poly_key, poly_base, d2.isoformat())

save_raw_json(p1, cfg.outputs_raw_dir / f'{pull_daily}_{d1}_grouped.json')
save_raw_json(p2, cfg.outputs_raw_dir / f'{pull_daily}_{d2}_grouped.json')

df1 = pd.DataFrame(p1.get('results', []) or [])
df2 = pd.DataFrame(p2.get('results', []) or [])

# Polygon grouped fields use short names
# T=ticker, c=close, v=volume
df1 = df1.rename(columns={'T':'symbol','c':'close_d1','v':'volume_d1'})
df2 = df2.rename(columns={'T':'symbol','c':'close_d2','v':'volume_d2'})

df_daily = df2[['symbol','close_d2','volume_d2']].merge(
    df1[['symbol','close_d1','volume_d1']], on='symbol', how='left'
)

df_daily['change_pct'] = ((df_daily['close_d2'] - df_daily['close_d1']) / df_daily['close_d1']) * 100
df_daily['provider'] = 'polygon'
df_daily['ts_utc'] = pd.Timestamp.now(tz=ZoneInfo('UTC'))
df_daily['session'] = 'regular'
df_daily['last_price'] = df_daily['close_d2']
df_daily['volume'] = df_daily['volume_d2']
df_daily['prev_close'] = df_daily['close_d1']
df_daily['source_endpoint'] = '/v2/aggs/grouped/locale/us/market/stocks/{date}'
df_daily['pull_id'] = pull_daily
df_daily['provider_payload'] = None

normalized_cols = ['provider','symbol','ts_utc','session','last_price','change_pct','volume','prev_close','source_endpoint','pull_id','provider_payload']
df_norm = df_daily[normalized_cols].dropna(subset=['symbol','last_price','volume','prev_close','change_pct'])

save_parquet(df_norm, cfg.outputs_processed_dir / f'{pull_daily}_polygon_grouped_normalized.parquet')

summary = quality_summary(df_norm)
summary


In [ ]:
top_movers = df_norm.sort_values('change_pct', ascending=False).head(50).copy()
top_movers[['symbol','change_pct','volume']].head(20)


In [ ]:
fig = px.histogram(df_norm, x='change_pct', nbins=100, title='DoD Change % Distribution (Grouped Daily)')
fig.show()

fig2 = px.histogram(df_norm, x='volume', nbins=100, title='Volume Distribution (Grouped Daily)')
fig2.update_xaxes(type='log')
fig2.show()


## 2) Polygon Intraday for Top Movers (1-minute bars)

In [ ]:
pull_intraday = new_pull_id('polygon_intraday')
end_date = d2
start_date = d1

symbols = top_movers['symbol'].head(20).tolist()
rows = []
for symbol in symbols:
    bars = polygon_intraday_bars(poly_key, poly_base, symbol, start_date.isoformat(), end_date.isoformat())
    save_raw_json(bars, cfg.outputs_raw_dir / f'{pull_intraday}_{symbol}_bars.json')
    for r in bars.get('results', []) or []:
        rows.append({
            'provider':'polygon',
            'symbol':symbol,
            'ts_utc': pd.to_datetime(r['t'], unit='ms', utc=True),
            'open': r.get('o'),
            'high': r.get('h'),
            'low': r.get('l'),
            'close': r.get('c'),
            'volume': r.get('v'),
            'source_endpoint':'/v2/aggs/ticker/{symbol}/range/1/minute/{from}/{to}',
            'pull_id': pull_intraday,
        })

df_intraday = pd.DataFrame(rows).sort_values(['symbol','ts_utc'])
if not df_intraday.empty:
    df_intraday['ret_1m_pct'] = df_intraday.groupby('symbol')['close'].pct_change() * 100

save_parquet(df_intraday, cfg.outputs_processed_dir / f'{pull_intraday}_intraday.parquet')
df_intraday.head()


## 3) Finnhub Quote Parity (free-tier accessible)

In [ ]:
if not fh_key:
    print('FINNHUB_API_KEY missing; skipping parity')
    df_parity = pd.DataFrame()
else:
    pull_parity = new_pull_id('parity_fh')
    frames = []
    for symbol in symbols[:10]:
        q = finnhub_quote(fh_key, fh_base, symbol)
        save_raw_json(q, cfg.outputs_raw_dir / f'{pull_parity}_{symbol}_finnhub_quote.json')
        frames.append(normalize_finnhub_quote(symbol, q, '/quote', pull_parity))

    df_fh = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    df_poly_cmp = df_norm[df_norm['symbol'].isin(df_fh['symbol'])][['symbol','last_price','change_pct','volume','prev_close']]
    df_fh_cmp = df_fh[['symbol','last_price','change_pct','volume','prev_close']]

    df_parity = df_poly_cmp.merge(df_fh_cmp, on='symbol', how='inner', suffixes=('_polygon','_finnhub'))
    if not df_parity.empty:
        df_parity['last_price_diff_pct'] = ((df_parity['last_price_finnhub'] - df_parity['last_price_polygon']) / df_parity['last_price_polygon']) * 100
        df_parity['change_pct_diff'] = df_parity['change_pct_finnhub'] - df_parity['change_pct_polygon']

    save_parquet(df_parity, cfg.outputs_processed_dir / f'{pull_parity}_parity.parquet')

df_parity.head()


## 4) Threshold Recommendations

In [ ]:
pct_candidates = [2,3,4,5,6,8,10]
vol_candidates = [50000,100000,200000,500000,1000000]

rows = []
for p in pct_candidates:
    for v in vol_candidates:
        m = df_norm[(df_norm['change_pct'] >= p) & (df_norm['volume'] >= v)]
        rows.append({
            'min_change_pct': p,
            'min_volume': v,
            'match_count': len(m),
            'median_change_pct': m['change_pct'].median() if not m.empty else None,
            'median_volume': m['volume'].median() if not m.empty else None,
        })

df_grid = pd.DataFrame(rows)
display(df_grid.head(40))

pivot = df_grid.pivot(index='min_volume', columns='min_change_pct', values='match_count')
fig_h = px.imshow(pivot, text_auto=True, title='Threshold Match Count Heatmap')
fig_h.show()

baseline = df_grid[(df_grid['min_change_pct']==4) & (df_grid['min_volume']==100000)]
print('baseline_recommendation')
display(baseline)
